# fase_4 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 4.

**Purpose**: Migrasi data Siswa dan Mitra dengan Mapping Kolom Spesifik

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import re
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_new']['database']}")

Connected to dataleap_v5_example and dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
hanif_tables_map = [
    ('siswa', 'siswa'),
    ('siswa_keluar', 'siswa_keluar'),
    ('mitra', 'mitra'),
    ('mitra_note', 'mitra_progres'),
    ('mitra_users', 'kemitraan_verifikator'),
    ('siswamitra', 'siswa_mitra'),
    ('siswa_keluar_mitra', 'siswa_mitra_keluar')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    try:
        cursor_old.execute(f"SELECT * FROM `{old_t}`")
        raw_data[old_t] = cursor_old.fetchall()
        print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")
    except Exception as e:
        print(f"❌ ERROR loading {old_t}: {e}")

✅ siswa loaded: 1469 records
✅ siswa_keluar loaded: 556 records
✅ mitra loaded: 22 records
✅ mitra_note loaded: 296 records
✅ mitra_users loaded: 228 records
✅ siswamitra loaded: 0 records
✅ siswa_keluar_mitra loaded: 0 records


## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [4]:
transformed_dfs = {}

# --- HELPER FUNCTIONS ---
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\\d+', str(s))
    return int(nums[0]) if nums else None

def extract_chars(s):
    if pd.isna(s) or not str(s).strip(): return None
    return re.sub(r'\\d+', '', str(s)).strip()

def convert_ya_tidak(s):
    if pd.isna(s): return 0
    val = str(s).strip().lower()
    return 1 if val == 'ya' else 0

# --- TRANSFORMATION ---

# 1. siswa -> siswa
if 'siswa' in raw_data:
    df = pd.DataFrame(raw_data['siswa'])
    df['id_mitra'] = df['idmitra'].apply(extract_int)
    
    # Normalisasi Agama
    agama_map = {
        'kristen': 'Kristen Protestan', 'protestan': 'Kristen Protestan', 
        'katholik': 'Katolik', 'budha': 'Buddha', 'khonghucu': 'Konghucu'
    }
    def normalize_agama(a):
        if pd.isna(a): return 'Islam'
        a_clean = str(a).strip()
        for k, v in agama_map.items():
            if k in a_clean.lower(): return v
        return a_clean
    df['agama'] = df['agama'].apply(normalize_agama)

    mapping = {
        'idsiswa': 'id_siswa', 'tgl_daftar': 'tanggal_registrasi', 'domisili': 'domisili',
        'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin',
        'nama_sekolah': 'asal_sekolah', 'level_sekolah': 'tingkat_sekolah', 'nama_ortu': 'nama_orang_tua',
        'pekerjaan_ortu': 'pekerjaan_orang_tua', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir',
        'no_induk': 'nomor_induk', 'email': 'email', 'idcalon': 'id_calon',
        'provinsi': 'id_provinsi', 'kabupaten': 'id_kabupaten', 'kecamatan': 'id_kecamatan',
        'kelurahan': 'id_kelurahan', 'id_mitra': 'id_mitra', 'nisn': 'nisn', 'nik': 'nik',
        'kewarganegaraan': 'kewarganegaraan', 'agama': 'agama', 'rt': 'rt', 'rw': 'rw',
        'kodepos': 'kode_pos', 'statussiswa': 'status_aktif', 'rekomen': 'rekomendasi',
        'info': 'sumber_info', 'pembayaran': 'metode_pembayaran', 'nama_ayah': 'nama_ayah',
        'pekerjaan_ayah': 'pekerjaan_ayah', 'jenjang_ayah': 'pendidikan_ayah', 
        'penghasilan_ayah': 'penghasilan_ayah', 'nama_ibu': 'nama_ibu', 'penghasilan_ibu': 'penghasilan_ibu',
        'jenjang_ibu': 'pendidikan_ibu', 'nama_wali': 'nama_wali', 'pekerjaan_wali': 'pekerjaan_wali',
        'jenjang_wali': 'pendidikan_wali', 'penghasilan_wali': 'penghasilan_wali',
        'wapeserta': 'wa_siswa', 'wawalmur': 'wa_ortu', 'waadmin': 'wa_administrasi',
        'sts_pengisian': 'status_pengisian', 'bukti': 'path_bukti_bayar', 'lulus': 'status_lulus_siswa',
        'created_bukti': 'tanggal_upload_bukti'
    }
    df_final = df.rename(columns=mapping)
    df_final['pekerjaan_ibu'] = None
    df_final['deleted_at'] = None
    target_cols = [c for c in list(mapping.values()) if c in df_final.columns] + ['pekerjaan_ibu', 'deleted_at']
    transformed_dfs['siswa'] = df_final[target_cols]

# 2. kursus_siswa
transformed_dfs['kursus_siswa'] = pd.DataFrame(columns=['id_kursus_siswa', 'id_siswa', 'id_kursus', 'tanggal_mulai', 'metode_belajar', 'status_aktif', 'catatan'])

# 3. siswa_keluar -> siswa_keluar
if 'siswa_keluar' in raw_data:
    df = pd.DataFrame(raw_data['siswa_keluar'])
    mapping = {
        'idsiswa_keluar': 'id_keluar', 'idsiswa': 'id_siswa',
        'alasan': 'alasan_keluar', 'tanggal': 'tanggal_keluar'
    }
    df = df.rename(columns=mapping)
    df['id_kursus'] = None
    df['id_tag_keluar'] = None
    transformed_dfs['siswa_keluar'] = df[list(mapping.values()) + ['id_kursus', 'id_tag_keluar']]

# 4. mitra -> mitra
if 'mitra' in raw_data:
    df = pd.DataFrame(raw_data['mitra'])
    df['id_mitra_new'] = df['idmitra'].apply(extract_int)
    df['kode_mitra'] = df['idmitra'].apply(extract_chars)
    
    bool_cols = ['leapverse', 'kemitraan', 'elsa', 'classin', 'mitraleap']
    for col in bool_cols:
        df[col] = df[col].apply(convert_ya_tidak)
        
    mapping = {
        'id_mitra_new': 'id_mitra', 'nama': 'nama_mitra', 'instansi': 'nama_instansi',
        'namasekolah': 'nama_sekolah', 'lokasi': 'alamat_mitra', 'kepsek': 'nama_pimpinan',
        'cp': 'kontak_mitra', 'status': 'status_mitra', 'visimisi': 'visi_misi',
        'program': 'program_mitra', 'sdm': 'info_sdm', 'weakness': 'info_kelemahan',
        'rekomen': 'rekomendasi_program', 'jenis': 'jenis_mitra', 'provinsi': 'provinsi_id',
        'kotkab': 'kabupaten_id', 'jml': 'jumlah_siswa_mitra', 'bidang': 'bidang_usaha',
        'leapverse': 'is_leapverse', 'kemitraan': 'status_kemitraan', 'tahun': 'tahun_bergabung',
        'jeniskemitraan': 'tipe_kerjasama', 'elsa': 'is_elsa', 'classin': 'is_classin',
        'mitraleap': 'is_mitra_leap', 'created_at': 'created_at', 'kode_mitra': 'kode_mitra'
    }
    transformed_dfs['mitra'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 5. mitra_note -> mitra_progres
if 'mitra_note' in raw_data:
    df = pd.DataFrame(raw_data['mitra_note'])
    mapping = {
        'idmnote': 'id_progres_mitra', 'idmitra': 'id_mitra',
        'note': 'catatan_progres_mitra', 'idusers': 'id_user', 'status': 'status_progres_mitra',
        'startdate': 'kemitraan_mulai', 'enddate': 'kemitraan_berakhir', 'created_at': 'created_at'
    }
    transformed_dfs['mitra_progres'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 6. mitra_users -> kemitraan_verifikator
if 'mitra_users' in raw_data:
    df = pd.DataFrame(raw_data['mitra_users'])
    mapping = {
        'idmusers': 'id_kemitraan', 'idmnote': 'id_progres_mitra', 'idusers': 'id_user'
    }
    transformed_dfs['kemitraan_verifikator'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 7. siswamitra -> siswa_mitra
if 'siswamitra' in raw_data:
    df = pd.DataFrame(raw_data['siswamitra'])
    mapping = {
        'idsiswa': 'id_sm', 'tgl_daftar': 'tanggal_daftar', 'domisili': 'alamat_domisili',
        'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin',
        'nama_instansi': 'nama_instansi', 'level_sekolah': 'tingkat_sekolah',
        'pekerjaan': 'pekerjaan_sm', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir',
        'no_induk': 'nomor_induk_sm', 'email': 'email_sm', 'tlp': 'wa_sm',
        'keluar': 'status_keluar_sm', 'idmitra': 'id_mitra'
    }
    df_final = df.rename(columns=mapping)
    df_final['sertifikat_sm'] = None
    transformed_dfs['siswa_mitra'] = df_final.reindex(columns=list(mapping.values()) + ['sertifikat_sm'])

# 8. siswa_keluar_mitra -> siswa_mitra_keluar
if 'siswa_keluar_mitra' in raw_data:
    df = pd.DataFrame(raw_data['siswa_keluar_mitra'])
    mapping = {
        'idsiswa_keluar': 'id_sm_keluar', 'idsiswa': 'id_sm',
        'alasan': 'alasan_keluar_sm', 'tanggal': 'tanggal_keluar_sm'
    }
    transformed_dfs['siswa_mitra_keluar'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

print(f"✓ Transformasi {len(transformed_dfs)} tabel Fase 4 selesai.")

✓ Transformasi 8 tabel Fase 4 selesai.


## 3.1 Verifikasi Hasil Transformasi
Bagian ini menampilkan perbandingan jumlah data dan tipe data untuk pengecekan manual.

In [ ]:
# 3.1.1 Ringkasan Jumlah Baris
print("📊 RINGKASAN MIGRASI (RECORDS COUNT)")
print("="*70)
summary_list = []
for old_t, new_t in hanif_tables_map:
    old_c = len(raw_data.get(old_t, []))
    new_c = len(transformed_dfs.get(new_t, []))
    summary_list.append({
        'Tabel Lama': old_t,
        'Tabel Baru': new_t,
        'Old Recs': old_c,
        'New Recs': new_c,
        'Diff': new_c - old_c,
        'Status': "✅ OK" if old_c == new_c else "⚠️ Cek"
    })
display(pd.DataFrame(summary_list))

total_old_all = sum(len(records) for records in raw_data.values())
total_new_all = sum(len(df) for df in transformed_dfs.values())
print(f"\n📢 TOTAL REKAPITULASI: {total_old_all} (Old) ➔ {total_new_all} (New)")
if total_old_all == total_new_all: print("✅ SEMUA DATA TERANGKUT")
else: print(f"⚠️ ADA SELISIH: {total_new_all - total_old_all} baris")

In [ ]:
# 3.1.2 Output Pengecekan Kolom Spesifik (KETERANGAN mapping.md)
print("\n🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)")
print("="*70)

# 1. Pengecekan ID Mitra (Siswa)
if 'siswa' in transformed_dfs:
    print("\n[SISWA] Pengecekan Ekstraksi ID Mitra (Int):")
    display(transformed_dfs['siswa'][['nama_lengkap', 'id_mitra']].dropna(subset=['id_mitra']).head(5))

# 2. Pengecekan Agama (Siswa)
if 'siswa' in transformed_dfs:
    print("\n[SISWA] Pengecekan Normalisasi Agama:")
    display(transformed_dfs['siswa']['agama'].value_counts())

# 3. Pengecekan Mitra (Boolean & ID/Kode)
if 'mitra' in transformed_dfs:
    print("\n[MITRA] Pengecekan Boolean (Ya/Tidak -> 1/0) & Kode Mitra:")
    display(transformed_dfs['mitra'][['nama_mitra', 'id_mitra', 'kode_mitra', 'is_leapverse', 'status_kemitraan']].head(5))

# 4. Pengecekan Audit Trailing (Mitra & Progres)
if 'mitra' in transformed_dfs:
    print("\n[MITRA] Pengecekan created_at (Direct Mapping):")
    display(transformed_dfs['mitra'][['nama_mitra', 'created_at']].head(5))

In [ ]:
# 3.1.3 Detail Perbandingan Kolom & Tipe Data (Side-by-Side)
print("\n🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE")
for old_t, new_t in hanif_tables_map:
    print(f"\n{'='*15} {old_t.upper()} ➔ {new_t.upper()} {'='*15}")
    
    df_old = pd.DataFrame(raw_data.get(old_t, []))
    df_new = transformed_dfs.get(new_t, pd.DataFrame())
    
    if not df_new.empty or not df_old.empty:
        comparison = []
        table_mapping = {}
        if old_t == 'siswa': table_mapping = {'idsiswa': 'id_siswa', 'tgl_daftar': 'tanggal_registrasi', 'domisili': 'domisili', 'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin', 'nama_sekolah': 'asal_sekolah', 'level_sekolah': 'tingkat_sekolah', 'nama_ortu': 'nama_orang_tua', 'pekerjaan_ortu': 'pekerjaan_orang_tua', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir', 'no_induk': 'nomor_induk', 'email': 'email', 'idcalon': 'id_calon', 'provinsi': 'id_provinsi', 'kabupaten': 'id_kabupaten', 'kecamatan': 'id_kecamatan', 'kelurahan': 'id_kelurahan', 'idmitra': 'id_mitra', 'nisn': 'nisn', 'nik': 'nik', 'kewarganegaraan': 'kewarganegaraan', 'agama': 'agama', 'rt': 'rt', 'rw': 'rw', 'kodepos': 'kode_pos', 'statussiswa': 'status_aktif', 'rekomen': 'rekomendasi', 'info': 'sumber_info', 'pembayaran': 'metode_pembayaran', 'nama_ayah': 'nama_ayah', 'pekerjaan_ayah': 'pekerjaan_ayah', 'jenjang_ayah': 'pendidikan_ayah', 'penghasilan_ayah': 'penghasilan_ayah', 'nama_ibu': 'nama_ibu', 'penghasilan_ibu': 'penghasilan_ibu', 'jenjang_ibu': 'pendidikan_ibu', 'nama_wali': 'nama_wali', 'pekerjaan_wali': 'pekerjaan_wali', 'jenjang_wali': 'pendidikan_wali', 'penghasilan_wali': 'penghasilan_wali', 'wapeserta': 'wa_siswa', 'wawalmur': 'wa_ortu', 'waadmin': 'wa_administrasi', 'sts_pengisian': 'status_pengisian', 'bukti': 'path_bukti_bayar', 'lulus': 'status_lulus_siswa', 'created_at': 'created_at'}
        elif old_t == 'siswa_keluar': table_mapping = {'idsiswa_keluar': 'id_keluar', 'idsiswa': 'id_siswa', 'alasan': 'alasan_keluar', 'tanggal': 'tanggal_keluar'}
        elif old_t == 'mitra': table_mapping = {'idmitra': 'id_mitra', 'nama': 'nama_mitra', 'instansi': 'nama_instansi', 'namasekolah': 'nama_sekolah', 'lokasi': 'alamat_mitra', 'kepsek': 'nama_pimpinan', 'cp': 'kontak_mitra', 'status': 'status_mitra', 'visimisi': 'visi_misi', 'program': 'program_mitra', 'sdm': 'info_sdm', 'weakness': 'info_kelemahan', 'rekomen': 'rekomendasi_program', 'jenis': 'jenis_mitra', 'provinsi': 'provinsi_id', 'kotkab': 'kabupaten_id', 'jml': 'jumlah_siswa_mitra', 'bidang': 'bidang_usaha', 'leapverse': 'is_leapverse', 'kemitraan': 'status_kemitraan', 'tahun': 'tahun_bergabung', 'jeniskemitraan': 'tipe_kerjasama', 'elsa': 'is_elsa', 'classin': 'is_classin', 'mitraleap': 'is_mitra_leap', 'created_at': 'created_at'}
        elif old_t == 'mitra_note': table_mapping = {'idmnote': 'id_progres_mitra', 'idmitra': 'id_mitra', 'note': 'catatan_progres_mitra', 'idusers': 'id_user', 'status': 'status_progres_mitra', 'startdate': 'kemitraan_mulai', 'enddate': 'kemitraan_berakhir', 'created_at': 'created_at'}
        elif old_t == 'mitra_users': table_mapping = {'idmusers': 'id_kemitraan', 'idmnote': 'id_progres_mitra', 'idusers': 'id_user'}
        elif old_t == 'siswamitra': table_mapping = {'idsiswa': 'id_sm', 'tgl_daftar': 'tanggal_daftar', 'domisili': 'alamat_domisili', 'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin', 'nama_instansi': 'nama_instansi', 'level_sekolah': 'tingkat_sekolah', 'pekerjaan': 'pekerjaan_sm', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir', 'no_induk': 'nomor_induk_sm', 'email': 'email_sm', 'tlp': 'wa_sm', 'keluar': 'status_keluar_sm', 'idmitra': 'id_mitra'}
        elif old_t == 'siswa_keluar_mitra': table_mapping = {'idsiswa_keluar': 'id_sm_keluar', 'idsiswa': 'id_sm', 'alasan': 'alasan_keluar_sm', 'tanggal': 'tanggal_keluar_sm'}

        for old_col, new_col in table_mapping.items():
            comparison.append({
                'Old Column': old_col,
                'Old Type': str(df_old[old_col].dtype) if not df_old.empty and old_col in df_old.columns else "N/A",
                '➔': '➔',
                'New Column': new_col,
                'New Type': str(df_new[new_col].dtype) if not df_new.empty and new_col in df_new.columns else "N/A"
            })
        
        # Cek kolom baru
        if not df_new.empty:
            for col in df_new.columns:
                if col not in table_mapping.values():
                    comparison.append({
                        'Old Column': '(KOLOM BARU / CUSTOM)',
                        'Old Type': '-',
                        '➔': '➔',
                        'New Column': col,
                        'New Type': str(df_new[col].dtype)
                    })
        
        display(pd.DataFrame(comparison))
        if not df_new.empty:
            print(f"\n--- SAMPLE DATA NEW (2 Baris) ---")
            display(df_new.head(2))
    else:
        print(f"⚠️ Tabel {new_t} kosong.")

## 4. Export ke Pickle

In [6]:
file_name = 'fase_4_hanif.pkl'

# Fix: Konversi tipe data StringDtype ke object agar kompatibel dengan Python 3.13 pickle
for table in transformed_dfs:
    df = transformed_dfs[table]
    if not df.empty:
        for col in df.columns:
            if str(df[col].dtype) in ['string', 'string[python]']:
                df[col] = df[col].astype(object)

pd.to_pickle(transformed_dfs, file_name)

total_records_new = sum(len(df) for df in transformed_dfs.values())
total_records_old = sum(len(records) for records in raw_data.values())

migration_result = {
    'fase': 'fase_4',
    'script': 'script_hanif',
    'fase_num': 4,
    'status': 'ready_for_insert',
    'old_records_total': total_records_old,
    'new_records_total': total_records_new,
    'diff': total_records_new - total_records_old,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()

{
  "fase": "fase_4",
  "script": "script_hanif",
  "fase_num": 4,
  "status": "ready_for_insert",
  "old_records_total": 2571,
  "new_records_total": 2571,
  "diff": 0,
  "pickle_file": "fase_4_hanif.pkl",
  "timestamp": "2026-04-29T11:12:35.877642"
}
